In [1]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from pyspark.sql.utils import AnalysisException

# Source catalog/schema
SOURCE_DB = "Stocks"
SOURCE_SCHEMA = "dbo"

# Lakehouse target
TARGET_SCHEMA = "dbo"
TARGET_TABLE_NAME = "DailyPortfolioPerformance"
TARGET_TABLE = f"{TARGET_SCHEMA}.{TARGET_TABLE_NAME}"

# Business key for incremental merge
# True account + stock grain 
BUSINESS_KEY = ["AccountId", "StockSymbol", "TradingDate"]

print("Target table  :", TARGET_TABLE)
print("Business key  :", BUSINESS_KEY)

# Initialize Spark session explicitly
#spark = SparkSession.builder.getOrCreate()

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 3, Finished, Available, Finished, False)

Target table  : dbo.DailyPortfolioPerformance
Business key  : ['AccountId', 'StockSymbol', 'TradingDate']


In [2]:
accounts      = spark.sql(f"SELECT * FROM `Link To Capacity East US Dev`.FinancialAdvising.dbo.Accounts")
account_stocks = spark.sql(f"SELECT * FROM `Link To Capacity East US Dev`.FinancialAdvising.dbo.AccountStocks")
stocks        = spark.sql(f"SELECT * FROM `Link To Capacity East US Dev`.FinancialAdvising.dbo.Stocks")
daily_stocks  = spark.sql(f"SELECT * FROM Stocks.dbo.dailystocks")

print("Source row counts:")
print("  Accounts       :", accounts.count())
print("  AccountStocks  :", account_stocks.count())
print("  Stocks         :", stocks.count())
print("  DailyStocks    :", daily_stocks.count())

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 4, Finished, Available, Finished, False)

Source row counts:
  Accounts       : 103
  AccountStocks  : 778
  Stocks         : 18
  DailyStocks    : 150002


In [3]:
df = spark.sql("SELECT * FROM Stocks.dbo.dailystocks")
#display(df)

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 5, Finished, Available, Finished, False)

In [4]:
accounts_df = accounts.select(
    F.col("Id").alias("AccountId"),
    F.col("AccountHolderFullName")
)

account_stocks_df = account_stocks.select(
    F.col("AccountId"),
    F.col("StockId"),
    F.col("Id").alias("PurchaseId"),
    F.col("PurchaseDateTime").alias("PurchaseDate"),
    F.col("PurchasePrice").alias("PurchaseSharePrice"),
    F.col("SharesPurchased"),
    F.col("InitialInvestment")
)

stocks_df = stocks.select(
    F.col("Id").alias("StockId"),
    F.col("StockSymbol"),
    F.col("CompanyName")
)

daily_stocks_df = daily_stocks.select(
    F.col("StockSymbol"),
    F.col("TradeDate"),
    F.col("Close")
)

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 6, Finished, Available, Finished, False)

In [5]:
aggregated_purchases = (
    accounts_df.alias("ac")
    .join(
        account_stocks_df.alias("acs"),
        F.col("ac.AccountId") == F.col("acs.AccountId"),
        "inner"
    )
    .join(
        stocks_df.alias("stc"),
        F.col("acs.StockId") == F.col("stc.StockId"),
        "inner"
    )
    .groupBy(
        F.col("ac.AccountId"),
        F.col("ac.AccountHolderFullName"),
        F.col("stc.StockSymbol"),
        F.col("stc.CompanyName")
    )
    .agg(
        # Earliest purchase date across all transactions for this stock
        F.min(F.col("acs.PurchaseDate")).alias("FirstPurchaseDate"),

        # Weighted average purchase price (price * shares / total shares)
        (
            F.sum(F.col("acs.PurchaseSharePrice") * F.col("acs.SharesPurchased")) /
            F.sum(F.col("acs.SharesPurchased"))
        ).alias("WeightedAvgPurchasePrice"),

        # Total shares across all transactions
        F.sum(F.col("acs.SharesPurchased")).alias("TotalSharesPurchased"),

        # Total initial investment across all transactions
        F.sum(F.col("acs.InitialInvestment")).alias("TotalInitialInvestment"),

        # Number of separate purchase transactions
        F.count(F.col("acs.PurchaseDate")).alias("NumberOfTransactions")
    )
)

print("Aggregated purchases row count:", aggregated_purchases.count())
display(aggregated_purchases.limit(10))

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 7, Finished, Available, Finished, False)

Aggregated purchases row count: 642


SynapseWidget(Synapse.DataFrame, c450b725-4a1c-4634-90d6-03be1024e664)

In [6]:
price_window = Window.partitionBy("StockSymbol").orderBy("TradeDate")
daily_window = Window.partitionBy(
    "StockSymbol",
    F.to_date("TradeDate")
).orderBy(F.col("TradeDate").desc())

daily_stock_history = (
    daily_stocks_df
    .select(
        F.col("StockSymbol"),
        F.col("TradeDate"),
        F.col("Close").alias("LastPrice"),
        F.lag("Close", 1).over(price_window).alias("PreviousDayPrice"),
        F.row_number().over(daily_window).alias("rn")
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

print("Daily stock history row count:", daily_stock_history.count())
display(daily_stock_history.limit(10))

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 8, Finished, Available, Finished, False)

Daily stock history row count: 92767


SynapseWidget(Synapse.DataFrame, 79e828ad-e3a7-4267-b274-c88de3cad53d)

In [7]:
daily_calculations = (
    aggregated_purchases.alias("asp")
    .join(
        daily_stock_history.alias("dsh"),
        (F.col("asp.StockSymbol") == F.col("dsh.StockSymbol")) &
        (F.col("dsh.TradeDate") >= F.col("asp.FirstPurchaseDate")),
        "inner"
    )
    .select(
        F.col("asp.AccountId"),
        F.col("asp.AccountHolderFullName"),
        F.col("asp.StockSymbol"),
        F.col("asp.CompanyName"),
        F.col("asp.FirstPurchaseDate"),
        F.col("asp.WeightedAvgPurchasePrice"),
        F.col("asp.TotalSharesPurchased"),
        F.col("asp.TotalInitialInvestment"),
        F.col("asp.NumberOfTransactions"),
        F.to_date(F.col("dsh.TradeDate")).alias("TradingDate"),
        F.col("dsh.LastPrice"),

        # Todays gain/loss per share
        (F.col("dsh.LastPrice") - F.col("dsh.PreviousDayPrice"))
            .alias("TodaysGainLossPerShare"),

        # Total todays gain/loss across all shares
        ((F.col("dsh.LastPrice") - F.col("dsh.PreviousDayPrice")) * F.col("asp.TotalSharesPurchased"))
            .alias("TodaysGainLoss"),

        # Current market value of entire holding
        (F.col("dsh.LastPrice") * F.col("asp.TotalSharesPurchased"))
            .alias("CurrentValue"),

        # Total gain/loss vs original investment
        ((F.col("dsh.LastPrice") * F.col("asp.TotalSharesPurchased")) - F.col("asp.TotalInitialInvestment"))
            .alias("TotalGainLoss"),

        # Todays gain/loss percentage
        F.when(
            (F.col("dsh.PreviousDayPrice").isNotNull()) & (F.col("dsh.PreviousDayPrice") != 0),
            ((F.col("dsh.LastPrice") - F.col("dsh.PreviousDayPrice")) / F.col("dsh.PreviousDayPrice")) 
        ).otherwise(F.lit(0.0)).alias("TodaysGainLossPercentage"),

        # Total gain/loss percentage vs original investment
        F.when(
            F.col("asp.TotalInitialInvestment") != 0,
            (((F.col("dsh.LastPrice") * F.col("asp.TotalSharesPurchased")) - F.col("asp.TotalInitialInvestment")) /
             F.col("asp.TotalInitialInvestment")) 
        ).otherwise(F.lit(0.0)).alias("TotalGainLossPercentage"),

        # Gain/loss vs weighted average purchase price
        F.when(
            F.col("asp.WeightedAvgPurchasePrice") != 0,
            ((F.col("dsh.LastPrice") - F.col("asp.WeightedAvgPurchasePrice")) /
             F.col("asp.WeightedAvgPurchasePrice")) 
        ).otherwise(F.lit(0.0)).alias("GainLossVsAvgCostPercentage")
    )
)

print("Daily calculations row count:", daily_calculations.count())
display(daily_calculations.limit(10))

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 9, Finished, Available, Finished, False)

Daily calculations row count: 59748


SynapseWidget(Synapse.DataFrame, 52231bf0-1ba2-4742-ac3e-695e9bd65a73)

In [8]:
account_day_window = Window.partitionBy("AccountId", "TradingDate")

final_df = (
    daily_calculations
    .withColumn(
        "PercentageOfAccount",
        (F.col("CurrentValue") / F.sum("CurrentValue").over(account_day_window)) 
    )
    .withColumn("ModifiedDate", F.current_timestamp())
)

print("Final row count:", final_df.count())
display(final_df.orderBy(F.col("TradingDate").desc(), F.col("AccountId")).limit(10))

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 10, Finished, Available, Finished, False)

Final row count: 59748


SynapseWidget(Synapse.DataFrame, 1b8cec28-b30f-4e82-9f81-b7e5969fc337)

In [9]:
dedupe_window = Window.partitionBy(
    *BUSINESS_KEY
).orderBy(
    F.col("FirstPurchaseDate").asc(),
    F.col("LastPrice").desc()
)

merge_source_df = (
    final_df
    .withColumn("rn_dedup", F.row_number().over(dedupe_window))
    .filter(F.col("rn_dedup") == 1)
    .drop("rn_dedup")
)

print("Merge source row count:", merge_source_df.count())

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 11, Finished, Available, Finished, False)

Merge source row count: 59748


In [10]:
dup_check = (
    merge_source_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
)

dup_count = dup_check.count()
print("Duplicate business keys found:", dup_count)

if dup_count > 0:
    display(dup_check)
    raise Exception(f"Duplicate rows found for merge key {BUSINESS_KEY}. Check source data.")

print("No duplicates found. Safe to merge.")

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 12, Finished, Available, Finished, False)

Duplicate business keys found: 0
No duplicates found. Safe to merge.


In [11]:


from delta.tables import DeltaTable

if not spark.catalog.tableExists(TARGET_TABLE):
    print(f"Delta table does not exist. Creating: {TARGET_TABLE}")
    (
        merge_source_df
        .withColumn("CreatedDate", F.current_timestamp())
        .select(
            "AccountId",
            "AccountHolderFullName",
            "StockSymbol",
            "CompanyName",
            "FirstPurchaseDate",
            "WeightedAvgPurchasePrice",
            "TotalSharesPurchased",
            "TotalInitialInvestment",
            "NumberOfTransactions",
            "TradingDate",
            "LastPrice",
            "TodaysGainLossPerShare",
            "TodaysGainLoss",
            "CurrentValue",
            "TotalGainLoss",
            "TodaysGainLossPercentage",
            "TotalGainLossPercentage",
            "GainLossVsAvgCostPercentage",
            "PercentageOfAccount",
            "CreatedDate",
            "ModifiedDate"
        )
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(TARGET_TABLE)
    )
    print(f"Created Delta table: {TARGET_TABLE}")

else:
    print(f"Delta table exists. Running incremental merge: {TARGET_TABLE}")
    delta_table = DeltaTable.forName(spark, TARGET_TABLE)

    (
        delta_table.alias("t")
        .merge(
            merge_source_df.alias("s"),
            """
            t.AccountId   = s.AccountId
            AND t.StockSymbol = s.StockSymbol
            AND t.TradingDate = s.TradingDate
            """
        )
        .whenMatchedUpdate(set={
            "AccountHolderFullName"       : "s.AccountHolderFullName",
            "CompanyName"                 : "s.CompanyName",
            "FirstPurchaseDate"           : "s.FirstPurchaseDate",
            "WeightedAvgPurchasePrice"    : "s.WeightedAvgPurchasePrice",
            "TotalSharesPurchased"        : "s.TotalSharesPurchased",
            "TotalInitialInvestment"      : "s.TotalInitialInvestment",
            "NumberOfTransactions"        : "s.NumberOfTransactions",
            "LastPrice"                   : "s.LastPrice",
            "TodaysGainLossPerShare"      : "s.TodaysGainLossPerShare",
            "TodaysGainLoss"              : "s.TodaysGainLoss",
            "CurrentValue"                : "s.CurrentValue",
            "TotalGainLoss"               : "s.TotalGainLoss",
            "TodaysGainLossPercentage"    : "s.TodaysGainLossPercentage",
            "TotalGainLossPercentage"     : "s.TotalGainLossPercentage",
            "GainLossVsAvgCostPercentage" : "s.GainLossVsAvgCostPercentage",
            "PercentageOfAccount"         : "s.PercentageOfAccount",
            "ModifiedDate"                : "s.ModifiedDate"
        })
        .whenNotMatchedInsert(values={
            "AccountId"                   : "s.AccountId",
            "AccountHolderFullName"       : "s.AccountHolderFullName",
            "StockSymbol"                 : "s.StockSymbol",
            "CompanyName"                 : "s.CompanyName",
            "FirstPurchaseDate"           : "s.FirstPurchaseDate",
            "WeightedAvgPurchasePrice"    : "s.WeightedAvgPurchasePrice",
            "TotalSharesPurchased"        : "s.TotalSharesPurchased",
            "TotalInitialInvestment"      : "s.TotalInitialInvestment",
            "NumberOfTransactions"        : "s.NumberOfTransactions",
            "TradingDate"                 : "s.TradingDate",
            "LastPrice"                   : "s.LastPrice",
            "TodaysGainLossPerShare"      : "s.TodaysGainLossPerShare",
            "TodaysGainLoss"              : "s.TodaysGainLoss",
            "CurrentValue"                : "s.CurrentValue",
            "TotalGainLoss"               : "s.TotalGainLoss",
            "TodaysGainLossPercentage"    : "s.TodaysGainLossPercentage",
            "TotalGainLossPercentage"     : "s.TotalGainLossPercentage",
            "GainLossVsAvgCostPercentage" : "s.GainLossVsAvgCostPercentage",
            "PercentageOfAccount"         : "s.PercentageOfAccount",
            "CreatedDate"                 : "current_timestamp()",
            "ModifiedDate"                : "s.ModifiedDate"
        })
        .execute()
    )

    print("Merge complete.")

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 13, Finished, Available, Finished, False)

Delta table exists. Running incremental merge: dbo.DailyPortfolioPerformance
Merge complete.


In [12]:
current_df = spark.table(TARGET_TABLE)

print("Current row count:", current_df.count())

summary = (
    current_df
    .groupBy("TradingDate")
    .agg(
        F.countDistinct("AccountId").alias("Accounts"),
        F.countDistinct("StockSymbol").alias("Stocks"),
        F.count("*").alias("Rows"),
        F.sum("CurrentValue").alias("TotalCurrentValue"),
        F.sum("TotalGainLoss").alias("TotalGainLoss")
    )
    .orderBy(F.col("TradingDate").desc())
)

display(summary)

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 14, Finished, Available, Finished, False)

Current row count: 59748


SynapseWidget(Synapse.DataFrame, 92c53a71-58e4-42df-a268-80751894c695)

In [13]:
history_df = spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE}")
display(history_df)

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 91718f3f-aaed-4aca-ad8b-df645f38e250)

In [14]:
VERSION_TO_READ = 0  # change to the version you want

historical_df = (
    spark.read
    .format("delta")
    .option("versionAsOf", VERSION_TO_READ)
    .table(TARGET_TABLE)
)

print(f"Row count at version {VERSION_TO_READ}:", historical_df.count())
display(historical_df.orderBy(F.col("TradingDate").desc()).limit(10))

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 16, Finished, Available, Finished, False)

Row count at version 0: 27998


SynapseWidget(Synapse.DataFrame, d1d910f7-e34b-447d-ac47-30266703de88)

In [15]:
TIMESTAMP_TO_READ = "2026-03-25T10:00:00Z"  # change to the date you want

historical_by_time_df = (
    spark.read
    .format("delta")
    .option("timestampAsOf", TIMESTAMP_TO_READ)
    .table(TARGET_TABLE)
)

print(f"Row count at {TIMESTAMP_TO_READ}:", historical_by_time_df.count())
display(historical_by_time_df.orderBy(F.col("TradingDate").desc()).limit(10))

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 17, Finished, Available, Finished, False)

Row count at 2026-03-25T10:00:00Z: 36344


SynapseWidget(Synapse.DataFrame, fa364e1f-b57e-4e70-9780-1119a4cee87b)

In [16]:
VERSION_A = 0
VERSION_B = 1

df_a = spark.read.format("delta").option("versionAsOf", VERSION_A).table(TARGET_TABLE)
df_b = spark.read.format("delta").option("versionAsOf", VERSION_B).table(TARGET_TABLE)

# Rows in B not in A (new inserts)
new_rows = df_b.exceptAll(df_a)
print(f"New rows added between v{VERSION_A} and v{VERSION_B}:", new_rows.count())
display(new_rows.limit(10))

# Rows in A not in B (deleted or changed)
changed_rows = df_a.exceptAll(df_b)
print(f"Rows changed or removed between v{VERSION_A} and v{VERSION_B}:", changed_rows.count())
display(changed_rows.limit(10))

StatementMeta(, 22cd4001-6e41-40f3-9c4f-766c9258c08d, 18, Finished, Available, Finished, False)

New rows added between v0 and v1: 36344


SynapseWidget(Synapse.DataFrame, 7f54a8c9-9d4b-431e-a480-5d3eb0826da3)

Rows changed or removed between v0 and v1: 27998


SynapseWidget(Synapse.DataFrame, 96c25bf9-d596-4c6b-9cb6-0e9b420ad5db)